# NYU Depth Training on Google Colab

**Runtime:** *Runtime → Change runtime type → T4 GPU*

## Efficient workflow (200k small files)

Google Drive is slow for hundreds of thousands of tiny PNGs. **Do not unzip on Drive.**

| Step | Where | Why |
|------|--------|-----|
| Store dataset | `My Drive/CV_Project/data/` | One-time upload/sync |
| **Best:** upload one archive | `nyu_dataset.tar.gz` in `data/` | 1 file instead of 200k uploads |
| Each Colab session | Extract/copy to `/content/nyu_data` | Colab local disk is fast for training I/O |
| Train | `NYU_DATA_ROOT=/content/nyu_data` | Loader reads from local disk |
| Save weights | `My Drive/CV_Project/checkpoints/` | Persists after session ends |

Your Drive layout:
```
CV_Project/
  data/
    nyu2_train.csv
    nyu2_test.csv
    nyu2_train/
    nyu2_test/
    nyu_dataset.tar.gz   # optional but recommended
  checkpoints/
```


## One-time setup on your Mac (before Colab)

Create a single archive from the four dataset items:

```bash
cd "/path/to/folder/with/csvs"
tar -czf nyu_dataset.tar.gz nyu2_train.csv nyu2_test.csv nyu2_train nyu2_test
```

Upload **only** `nyu_dataset.tar.gz` into `My Drive/CV_Project/data/` via Drive for desktop.

> If you already synced the extracted folders to Drive, skip the tar and use **MODE = "rsync"** below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# --- EDIT if your Drive path differs ---
DRIVE_PROJECT = Path("/content/drive/MyDrive/CV_Project")
DRIVE_DATA = DRIVE_PROJECT / "data"
DRIVE_CHECKPOINTS = DRIVE_PROJECT / "checkpoints"

LOCAL_DATA = Path("/content/nyu_data")
LOCAL_REPO = Path("/content/cv_project")

# "tar" = extract nyu_dataset.tar.gz to LOCAL_DATA (recommended)
# "rsync" = copy already-extracted files from DRIVE_DATA to LOCAL_DATA
MODE = "tar"
TAR_NAME = "nyu_dataset.tar.gz"

print("Drive project:", DRIVE_PROJECT)
print("Drive data:", DRIVE_DATA)
print("Drive checkpoints:", DRIVE_CHECKPOINTS)


In [ ]:
from colab_setup import stage_from_rsync, stage_from_tar, verify_dataset

if MODE == "tar":
    tar_path = DRIVE_DATA / TAR_NAME
    if not tar_path.exists():
        raise FileNotFoundError(
            f"{tar_path} not found. Upload nyu_dataset.tar.gz to CV_Project/data/ "
            "or set MODE='rsync' if folders are already on Drive."
        )
    DATA_ROOT = stage_from_tar(tar_path, LOCAL_DATA)
elif MODE == "rsync":
    DATA_ROOT = stage_from_rsync(DRIVE_DATA, LOCAL_DATA)
else:
    raise ValueError("MODE must be 'tar' or 'rsync'")

print("Training will use:", DATA_ROOT)
verify_dataset(DATA_ROOT)
print("Dataset OK")


In [ ]:
import os

os.environ["NYU_DATA_ROOT"] = str(DATA_ROOT)
os.environ["CHECKPOINT_DIR"] = str(DRIVE_CHECKPOINTS)

DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("NYU_DATA_ROOT:", os.environ["NYU_DATA_ROOT"])
print("CHECKPOINT_DIR:", os.environ["CHECKPOINT_DIR"])


In [ ]:
import subprocess
import os

REPO_URL = "https://github.com/YOUR_USERNAME/cv_project.git"

if not LOCAL_REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)

os.chdir(LOCAL_REPO)
print("Working directory:", os.getcwd())


In [ ]:
!pip install -q opencv-python-headless tqdm matplotlib


In [ ]:
!python smoke_train.py


In [ ]:
import config

# ~8k random train frames per epoch (~5 min) vs 50k (~30 min) on T4
config.MAX_TRAIN_SAMPLES = 8000
config.BATCH_SIZE = 16
config.EPOCHS = 15
config.NUM_WORKERS = 2

# Full training (slow): config.MAX_TRAIN_SAMPLES = None

!python train.py


## Checkpoints during training

`train.py` validates on the **full 654-image test split** every epoch and saves **`best_unet.pt`** whenever validation RMSE improves (to `CV_Project/checkpoints/` on Drive).

You will see `saved best -> ...` in the epoch log when it updates.


In [ ]:
import os

os.environ["NYU_DATA_ROOT"] = str(DATA_ROOT)
os.environ["CHECKPOINT_DIR"] = str(DRIVE_CHECKPOINTS)

ckpt = DRIVE_CHECKPOINTS / "best_unet.pt"
print("Evaluating:", ckpt)

!python evaluate.py --checkpoint "{ckpt}"



Run this cell after training (or in a fresh session after staging data + setting env vars) to print final test metrics and show a sample prediction plot.
